### Simple Neural Network

In [1]:
nn_architecture = [
    {"input_dim": 2, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 6, "activation": "relu"},
    {"input_dim": 6, "output_dim": 6, "activation": "relu"},
    {"input_dim": 6, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 1, "activation": "sigmoid"},
]

In [18]:
import numpy as np
def init_layers(nn_architecture, seed = 99):
    np.random.seed(seed)
    number_of_layers = len(nn_architecture)
    params_values = {}

    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        layer_input_size = layer["input_dim"]
        layer_output_size = layer['output_dim']

        params_values["W" + str(layer_idx)] = np.random.randn(layer_output_size, layer_input_size) * 0.1
        params_values['b' + str(layer_idx)] = np.random.randn(layer_output_size, 1) * 0.1

    return params_values

In [3]:
"""
Activation functions used by layers
First two are forward propagation proccess
While last two are derivative functions (gradients) of the first two, used in backpropagation 
"""

def sigmoid(Z):
    return 1/(1+np.exp(-Z))

def relu(Z):
    return np.maximum(0,Z)

def sigmoid_backward(dA, Z):
    sig = sigmoid(Z)
    return dA * sig * (1 - sig)

def relu_backward(dA, Z):
    dZ = np.array(dA, copy = True)
    dZ[Z <= 0] = 0
    return dZ

In [4]:
"""
Function for single forward propagation step.

Parameters:
- A_prev: numpy array
    Activations from the previous layer (input to the current layer).
- W_curr: numpy array
    Weights applied to the current layer.
- b_curr: numpy array
    Bias for the current layer.
- activation: str
    Activation function to be used ("relu" or "sigmoid").

Returns:
- A_curr: numpy array
    Activations from the current layer after applying the activation function.
- Z_curr: numpy array
    Linear combination of inputs and weights before applying the activation function (basically it's applying weight value to the input value).
"""

def single_layer_forward_propagation(A_prev, W_curr, b_curr, activation="relu"):
    Z_curr = np.dot(W_curr, A_prev) + b_curr
    
    if activation == "relu":
        activation_func = relu
    elif activation == "sigmoid":
        activation_func = sigmoid
    else:
        raise Exception('Non-supported activation function')
        
    return activation_func(Z_curr), Z_curr

In [5]:
"""
Full forward propagation function utilizing single step forward propagation function.

Parameters:
- X: numpy array
    Input data (features) for the neural network.
- params_values: dict
    Dictionary containing the weights and biases for each layer (e.g. W1, b2...).
- nn_architecture: list of dicts
    List of dictionaries defining the neural network architecture, including input/output sizes and activation functions for each layer. Defined above

Returns:
- A_curr: numpy array
    Final activations after passing through all layers.
- memory: dict
    Dictionary storing activations and linear combinations for each layer.
"""

def full_forward_propagation(X, params_values, nn_architecture):
    memory = {}
    A_curr = X
    
    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1 # Index of current layer (added 1 to turn index to one-based index)
        A_prev = A_curr # Prev layer (which is input to the single layer forward function)
        
        activ_function_curr = layer["activation"]
        W_curr = params_values["W" + str(layer_idx)] # Get Weight matrix of currect layer (e.g. W1)
        b_curr = params_values["b" + str(layer_idx)]
        A_curr, Z_curr = single_layer_forward_propagation(A_prev, W_curr, b_curr, activ_function_curr) # Calling single layer propagation, getting Current activation value 
        
        # Adding values to memory dictionary
        memory["A" + str(idx)] = A_prev
        memory["Z" + str(layer_idx)] = Z_curr
       
    return A_curr, memory

In [6]:
def get_cost_value(Y_hat, Y):
    m = Y_hat.shape[1]
    cost = -1 / m * (np.dot(Y, np.log(Y_hat).T) + np.dot(1 - Y, np.log(1 - Y_hat).T))
    return np.squeeze(cost)

def convert_prob_into_class(probs):
    probs_ = np.copy(probs)
    probs_[probs_ > 0.5] = 1
    probs_[probs_ <= 0.5] = 0
    return probs_

def get_accuracy_value(Y_hat, Y):
    Y_hat_ = convert_prob_into_class(Y_hat)
    return (Y_hat_ == Y).all(axis=0).mean()

In [7]:
"""
Performs the backward propagation for a single layer in a neural network.

Parameters:
- dA_curr: numpy array
    Gradient of the loss with respect to the activations of the current layer.
- W_curr: numpy array
    Weights of the current layer.
- b_curr: numpy array
    Biases of the current layer.
- Z_curr: numpy array
    Linear combination of inputs and weights before applying the activation function in the current layer.
- A_prev: numpy array
    Activations from the previous layer (input to the current layer).
- activation: str, optional
    Activation function used in the current layer ("relu" or "sigmoid"). Default is "relu".

Returns:
- dA_prev: numpy array
    Gradient of the loss with respect to the activations from the previous layer.
- dW_curr: numpy array
    Gradient of the loss with respect to the weights of the current layer.
- db_curr: numpy array
    Gradient of the loss with respect to the biases of the current layer.
"""
def single_layer_backward_propagation(dA_curr, W_curr, b_curr, Z_curr, A_prev, activation="relu"):
    m = A_prev.shape[1]
    
    # Get activation function
    if activation is "relu":
        backward_activation_func = relu_backward
    elif activation is "sigmoid":
        backward_activation_func = sigmoid_backward
    else:
        raise Exception('Non-supported activation function')
    
    # Get gradient of loss function with respect to Z of current layer
    dZ_curr = backward_activation_func(dA_curr, Z_curr)
    # Compute gradient of loss with respect to weights of current layer
    dW_curr = np.dot(dZ_curr, A_prev.T) / m
    # Compute gradient of loss iwht respect of biases of current layer
    db_curr = np.sum(dZ_curr, axis=1, keepdims=True) / m

    # Compute gradient of loss with respect to activation from previous layer (dZ_curr)
    dA_prev = np.dot(W_curr.T, dZ_curr)

    return dA_prev, dW_curr, db_curr

<>:30: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:32: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:30: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:32: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
C:\Users\augus\AppData\Local\Temp\ipykernel_30032\4172553760.py:30: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if activation is "relu":
C:\Users\augus\AppData\Local\Temp\ipykernel_30032\4172553760.py:32: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif activation is "sigmoid":


In [8]:

"""
Performs the backward propagation through the entire neural network.

Parameters:
- Y_hat: numpy array
    Predicted probabilities (output of the neural network).
- Y: numpy array
    True labels.
- memory: dict
    Dictionary storing activations and linear combinations (Z) for each layer from the forward propagation step.
- params_values: dict
    Dictionary containing the weights and biases for each layer.
- nn_architecture: list of dicts
    List of dictionaries defining the neural network architecture, including input/output sizes and activation functions for each layer.

Returns:
- grads_values: dict
    Dictionary containing the gradients of the loss with respect to the weights and biases for each layer.
"""
def full_backward_propagation(Y_hat, Y, memory, params_values, nn_architecture):
    grads_values = {}
    m = Y.shape[1]
    Y = Y.reshape(Y_hat.shape)
   
   # Compute the initial gradient of the loss with respect to the activations of the output layer
    dA_prev = - (np.divide(Y, Y_hat) - np.divide(1 - Y, 1 - Y_hat))
    
    # Loop through each layer in reverse order (from output layer to input layer)
    for layer_idx_prev, layer in reversed(list(enumerate(nn_architecture))):
        layer_idx_curr = layer_idx_prev + 1
        activ_function_curr = layer["activation"]
        
        dA_curr = dA_prev # Gradient of the loss with respect to the activations of the current layer
        
        # Getting values from memory object retunred from forward propagation
        A_prev = memory["A" + str(layer_idx_prev)]
        Z_curr = memory["Z" + str(layer_idx_curr)]
        W_curr = params_values["W" + str(layer_idx_curr)]
        b_curr = params_values["b" + str(layer_idx_curr)]
        
        # Perform backward propagation for the current layer
        dA_prev, dW_curr, db_curr = single_layer_backward_propagation(
            dA_curr, W_curr, b_curr, Z_curr, A_prev, activ_function_curr)
        
        # Store the gradients of the weights and biases for the current layer
        grads_values["dW" + str(layer_idx_curr)] = dW_curr
        grads_values["db" + str(layer_idx_curr)] = db_curr
    
    return grads_values

In [20]:

def update(params_values, grads_values, nn_architecture, learning_rate):
    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        params_values["W" + str(layer_idx)] -= learning_rate * grads_values["dW" + str(layer_idx)]        
        params_values["b" + str(layer_idx)] -= learning_rate * grads_values["db" + str(layer_idx)]

    return params_values

In [10]:
def train(X, Y, nn_architecture, epochs, learning_rate):
    params_values = init_layers(nn_architecture, 2)
    cost_history = []
    accuracy_history = []
    
    for i in range(epochs):
        Y_hat, cashe = full_forward_propagation(X, params_values, nn_architecture)
        cost = get_cost_value(Y_hat, Y)
        cost_history.append(cost)
        accuracy = get_accuracy_value(Y_hat, Y)
        accuracy_history.append(accuracy)
        
        grads_values = full_backward_propagation(Y_hat, Y, cashe, params_values, nn_architecture)
        params_values = update(params_values, grads_values, nn_architecture, learning_rate)
        
    return params_values, cost_history, accuracy_history


In [13]:
import os
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
sns.set_style("whitegrid")

%pip install tensorflow
%pip install keras

import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical
from keras import regularizers

from sklearn.metrics import accuracy_score

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
# number of samples in the data set
N_SAMPLES = 1000
# ratio between training and test sets
TEST_SIZE = 0.1

X, y = make_moons(n_samples = N_SAMPLES, noise=0.2, random_state=100)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=42)

In [25]:
params_values = train(np.transpose(X_train), np.transpose(y_train.reshape((y_train.shape[0], 1))), nn_architecture, 10000, 0.01)[0]


In [26]:
Y_test_hat, _ = full_forward_propagation(np.transpose(X_test), params_values, nn_architecture)

In [27]:
acc_test = get_accuracy_value(Y_test_hat, np.transpose(y_test.reshape((y_test.shape[0], 1))))
print("Test set accuracy: {:.2f} - David".format(acc_test))

Test set accuracy: 0.46 - David
